# Experiment: Module 11A — Native Mac MPS prediction evidence

**Objective.** Re-run the immutable seed-42 LoRA checkpoint on the Mac's real MPS device and create a hash-bound, metadata-only reference report for a later CUDA comparison.

**Success criteria.** Explicit MPS selection must fail if unavailable; all 12 registered synthetic cases must complete; no message text may enter the report; no official BANKING77 test data may be loaded. This experiment evaluates execution parity, not model quality.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (current, *current.parents) if (path / 'pyproject.toml').is_file())
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SEED = 42
'Repository-local environment initialised'

'Repository-local environment initialised'

## Registered plan

1. Parse and hash-check the explicit MPS runtime profile.
2. Execute a small real-device tensor probe and record hardware/software metadata.
3. Load the exact historical adapter, calibration report, label order and base-model revision.
4. Run only the existing synthetic API shadow fixture after PII redaction.
5. Validate the self-hashing report and privacy/data boundary.

`PYTORCH_ENABLE_MPS_FALLBACK=1` is prohibited, so an unsupported Metal operator cannot silently run on CPU.

In [2]:
from governed_banking.runtime_evidence import RuntimeProfile, run_runtime_verification

assert os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK') != '1'
mps_profile = RuntimeProfile.from_yaml(PROJECT_ROOT / 'configs/runtime/mps.yaml')
runtime_report = run_runtime_verification(
    mps_profile,
    report_path=PROJECT_ROOT / 'reports/runtime/mps-runtime.json',
    implementation_paths={
        'accelerator.py': PROJECT_ROOT / 'src/governed_banking/accelerator.py',
        'runtime_evidence.py': PROJECT_ROOT / 'src/governed_banking/runtime_evidence.py',
        'verify_accelerator.py': PROJECT_ROOT / 'scripts/verify_accelerator.py',
    },
    seed=SEED,
)
{
    'selected': runtime_report['runtime']['selected'],
    'accelerator': runtime_report['runtime']['accelerator_name'],
    'torch': runtime_report['runtime']['torch_version'],
    'report_sha256': runtime_report['report_sha256'],
}

{'selected': 'mps',
 'accelerator': 'Apple M4',
 'torch': '2.13.0',
 'report_sha256': '9c59a4f7dac4a36091c5a1fde031fcd6c50da3f7b02064e16fcf5bc6e450248f'}

In [3]:
from governed_banking.parity import PredictionParityConfig, run_backend_evidence

parity_config = PredictionParityConfig.from_yaml(PROJECT_ROOT / 'configs/prediction_parity.yaml')
mps_report = run_backend_evidence(
    parity_config,
    backend='mps',
    report_path=PROJECT_ROOT / 'reports/parity/mps-seed42.json',
    implementation_paths={
        'accelerator.py': PROJECT_ROOT / 'src/governed_banking/accelerator.py',
        'parity.py': PROJECT_ROOT / 'src/governed_banking/parity.py',
        'portable_inference.py': PROJECT_ROOT / 'src/governed_banking/portable_inference.py',
        'run_backend_parity.py': PROJECT_ROOT / 'scripts/run_backend_parity.py',
    },
)
{
    'backend': mps_report['backend'],
    'case_count': mps_report['case_count'],
    'report_sha256': mps_report['report_sha256'],
}

/Users/coded/Downloads/IAPP-Artificial-Intelligence-Governance-Professional-AIGP-Complete-Course/AI Engineer/governed-banking-intent-router/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19888.74it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: /Users/coded/Downloads/IAPP-Artificial-Intelligence-Governance-Professional-AIGP-Complete-Course/AI Engineer/governed-banking-intent-router/artifacts/huggingface/models--FacebookAI--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weig

{'backend': 'mps',
 'case_count': 12,
 'report_sha256': '9373e1c1193304bbb9341f9664de855a82ac0ca56687ab4aef49e2e8f2050667'}

## Evidence validation

The checks below fail the notebook if the requested device was changed, if the report retained text, if its self-hash is invalid, or if the run crossed the registered data boundary.

In [4]:
from governed_banking.parity import validate_backend_report
from governed_banking.runtime_evidence import validate_runtime_report

validate_runtime_report(runtime_report, profile=mps_profile)
validate_backend_report(mps_report, config=parity_config, expected_backend='mps')
assert mps_report['runtime']['selected'] == 'mps'
assert mps_report['data_boundary'] == {
    'fixture_is_synthetic': True,
    'input_text_persisted': False,
    'redacted_text_persisted': False,
    'official_test_access': False,
    'customer_data_access': False,
}
'All registered MPS evidence gates passed'

'All registered MPS evidence gates passed'

## Result and next step

A passing run creates `reports/runtime/mps-runtime.json` and `reports/parity/mps-seed42.json`. These are the immutable Mac reference artifacts. Run Notebook 11B on a real Google Colab CUDA GPU; it will generate independent CUDA evidence and compare it with this MPS report. Do not interpret parity as evidence of production fitness or improved classification performance.